# 特质性波动率异象 (IVOL Anomaly) 完整教程：从残差计算到异象检验

## 📚 教学目标
1. 理解特质波动率 (IVOL) 的定义：CAPM/FF3 回归残差的标准差
2. 掌握 IVOL 的计算方法：时序回归 → 残差 → 标准差
3. 在微型数据集（10 只股票）上手算 IVOL
4. 扩展到完整规模（300 只 × 60 月），检验 IVOL 异象（低 IVOL 组合表现更好）
5. 理解 IVOL 异象的行为金融学解释（lottery demand、套利不对称性）

**参考书**：《因子投资：方法与实践》（石川）第 5.3 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：特质波动率之谜

### 🎯 核心问题

经典金融理论（CAPM）告诉我们：股票的风险可以分解为**系统性风险**（由因子代表）和**特质性风险**（idiosyncratic risk）。

由于特质性风险可以通过分散化投资消除，投资者**不应该**因为承担特质性风险而获得补偿。换言之：

> **特质波动率 (IVOL) 与预期收益率应该无关。**

然而，Ang et al. (2006) 发表在 *Journal of Finance* 上的经典论文发现：

> **IVOL 高的股票，未来收益率反而更低！** 即 IVOL 与收益率呈**负相关**。

这就是著名的**特质性波动率之谜 (Idiosyncratic Volatility Puzzle)**。

### 📖 书中的定义

> 由多因子定价模型可知，股票的风险可以分解为因子代表的系统性风险以及特质性风险两部分。后者通常通过股票收益率的特质性波动率（idiosyncratic volatility）来度量。

### 🔗 IVOL 计算流程

```
个股超额收益率 + 因子收益率
        ↓
   时序回归 (OLS)
        ↓
   提取残差 ε
        ↓
   σ(ε) = IVOL
        ↓
   按 IVOL 分组 → 检验异象
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. IVOL 的计算公式

### 📐 CAPM 模型下的 IVOL

对每只股票 $i$，使用过去 $T$ 个交易日的数据进行时序回归：

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i (r_{m,t} - r_{f,t}) + \varepsilon_{i,t}$$

其中：
- $r_{i,t} - r_{f,t}$ = 股票 $i$ 在第 $t$ 日的超额收益率
- $r_{m,t} - r_{f,t}$ = 市场超额收益率
- $\beta_i$ = 市场暴露（beta）
- $\varepsilon_{i,t}$ = 残差收益率（特质性收益）

### 📐 IVOL 的定义

$$\text{IVOL}_i = \sqrt{\frac{1}{T-2} \sum_{t=1}^{T} \hat{\varepsilon}_{i,t}^2}$$

即：回归残差的样本标准差。

### 📐 FF3 模型下的 IVOL

使用 Fama-French 三因子模型替代 CAPM：

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i^{MKT}(r_{m,t} - r_{f,t}) + \beta_i^{SMB} \cdot SMB_t + \beta_i^{HML} \cdot HML_t + \varepsilon_{i,t}$$

此时残差的标准差即为基于 FF3 的 IVOL。本书实证中采用 FF3 模型。

---

## 3. 微型数据集：手算 IVOL（10 只股票 × 21 日）

### 📊 生成模拟数据

我们先用 10 只股票、21 个交易日的微型数据，手算每只股票的 IVOL。

数据生成过程 (DGP)：
- 市场超额收益率：$r_m - r_f \sim N(0.05\%, 1.5\%)$ 日频
- 个股超额收益率：$r_i - r_f = \alpha_i + \beta_i (r_m - r_f) + \varepsilon_i$
- 残差 $\varepsilon_i \sim N(0, \sigma_i)$，其中 $\sigma_i$ 在股票间不同（模拟不同 IVOL）

In [ ]:
# ========== 步骤 1: 生成微型数据 ==========
N_STOCKS = 10
T_DAYS = 21  # 约一个月的交易日

# 市场超额收益率
mkt_excess = np.random.normal(0.0005, 0.015, T_DAYS)

# 每只股票的参数
betas = np.array([0.8, 1.0, 1.2, 0.9, 1.1, 1.3, 0.7, 1.0, 1.5, 0.6])
ivols_true = np.array([0.01, 0.02, 0.03, 0.015, 0.025, 0.035, 0.012, 0.02, 0.04, 0.008])
alphas = np.zeros(N_STOCKS)  # 真实 alpha = 0

# 生成个股超额收益率
stock_returns = np.zeros((T_DAYS, N_STOCKS))
residuals_true = np.zeros((T_DAYS, N_STOCKS))

for i in range(N_STOCKS):
    eps = np.random.normal(0, ivols_true[i], T_DAYS)
    residuals_true[:, i] = eps
    stock_returns[:, i] = alphas[i] + betas[i] * mkt_excess + eps

print("📊 步骤 1: 微型数据生成完成")
print(f"  股票数量: {N_STOCKS}")
print(f"  交易日数: {T_DAYS}")
print(f"  真实 IVOL: {ivols_true}")
print(f"  真实 Beta: {betas}")

### 📐 手算 IVOL：以股票 0 为例

对股票 0 进行 CAPM 回归：

$$r_{0,t} - r_f = \alpha_0 + \beta_0 (r_{m,t} - r_f) + \varepsilon_{0,t}$$

然后计算 $\text{IVOL}_0 = \text{std}(\hat{\varepsilon}_{0,t})$

In [ ]:
# ========== 步骤 2: 手算股票 0 的 IVOL ==========
stock_idx = 0
y = stock_returns[:, stock_idx]
X = sm.add_constant(mkt_excess)

# OLS 回归
model = sm.OLS(y, X).fit()
alpha_hat = model.params[0]
beta_hat = model.params[1]
residuals = model.resid

# IVOL = 残差的标准差
ivol_manual = np.std(residuals, ddof=2)  # 自由度调整: T - 2

print(f"📊 步骤 2: 股票 {stock_idx} 的 IVOL 计算")
print(f"  回归截距 α̂ = {alpha_hat:.6f}")
print(f"  回归斜率 β̂ = {beta_hat:.4f}")
print(f"  残差 ε: 均值 = {np.mean(residuals):.8f}, 标准差 = {ivol_manual:.6f}")
print(f"  手算 IVOL = {ivol_manual:.6f}")
print(f"  真实 IVOL = {ivols_true[stock_idx]:.6f}")
print(f"  💡 手算值与真实值接近，回归成功恢复了 IVOL")

In [ ]:
# ========== 步骤 3: 计算全部 10 只股票的 IVOL ==========
ivols_estimated = []
betas_estimated = []

print("📊 步骤 3: 全部股票 IVOL 计算")
print(f"{'股票':>4} {'β̂':>8} {'IVOL_估计':>10} {'IVOL_真实':>10} {'残差':>8}")
print("-" * 50)

for i in range(N_STOCKS):
    y_i = stock_returns[:, i]
    X_i = sm.add_constant(mkt_excess)
    model_i = sm.OLS(y_i, X_i).fit()
    
    ivol_i = np.std(model_i.resid, ddof=2)
    ivols_estimated.append(ivol_i)
    betas_estimated.append(model_i.params[1])
    
    print(f"  {i:>2} {model_i.params[1]:>8.4f} {ivol_i:>10.6f} {ivols_true[i]:>10.6f} {ivol_i - ivols_true[i]:>+8.6f}")

ivols_estimated = np.array(ivols_estimated)
betas_estimated = np.array(betas_estimated)

print(f"\n💡 估计值与真实值的相关系数: {np.corrcoef(ivols_estimated, ivols_true)[0,1]:.4f}")
print(f"  💡 21 个数据点的估计有噪声，但趋势一致")

In [ ]:
# ========== 用 scipy 验证 OLS 回归 ==========
# 对股票 0 用 scipy 的 linregress 验证
slope, intercept, r_value, p_value, std_err = stats.linregress(mkt_excess, stock_returns[:, 0])

print("🔬 scipy.stats.linregress 验证 (股票 0):")
print(f"  手算 α̂  = {alpha_hat:.8f}")
print(f"  scipy α̂ = {intercept:.8f}")
print(f"  手算 β̂  = {beta_hat:.6f}")
print(f"  scipy β̂ = {slope:.6f}")
print(f"\n  ✅ 验证通过！")

In [ ]:
# ========== 可视化: 估计 IVOL vs 真实 IVOL ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 估计 vs 真实
ax = axes[0]
ax.scatter(ivols_true, ivols_estimated, s=100, c='steelblue', edgecolors='black', zorder=5)
for i in range(N_STOCKS):
    ax.annotate(f'S{i}', (ivols_true[i], ivols_estimated[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=9)
min_val = min(ivols_true.min(), ivols_estimated.min()) * 0.8
max_val = max(ivols_true.max(), ivols_estimated.max()) * 1.2
ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7, label='45-degree line')
ax.set_xlabel('True IVOL', fontsize=12)
ax.set_ylabel('Estimated IVOL', fontsize=12)
ax.set_title('Estimated vs True IVOL (10 Stocks)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# 右图: 残差分布 (股票 0)
ax = axes[1]
ax.hist(residuals, bins=8, density=True, alpha=0.7, color='steelblue', edgecolor='black')
x_range = np.linspace(residuals.min(), residuals.max(), 100)
ax.plot(x_range, stats.norm.pdf(x_range, 0, ivols_true[0]), 'r-', lw=2,
        label=f'N(0, {ivols_true[0]:.4f})')
ax.set_xlabel('Residual (ε)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Residual Distribution - Stock 0 (IVOL={ivol_manual:.4f})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图：估计 IVOL 与真实 IVOL 的散点图，点越接近 45 度线越准确")
print("  右图：股票 0 的残差分布直方图，叠加理论正态分布曲线")

---

## 4. 扩展到完整规模：300 只股票 × 60 个月

### 📊 数据生成过程 (DGP)

现在扩展到更真实的规模：
- 300 只股票
- 60 个月（5 年）
- 每月 21 个交易日用于计算 IVOL
- 因子模型：Fama-French 三因子

**关键设计**：IVOL 与未来收益率呈**负相关**（模拟 IVOL 异象）
- 高 IVOL 股票的预期月收益率更低
- 低 IVOL 股票的预期月收益率更高

In [ ]:
# ========== 步骤 4: 完整规模模拟参数 ==========
N_STOCKS_FULL = 300
N_MONTHS = 60
N_DAYS_PER_MONTH = 21
N_GROUPS = 5  # IVOL 分 5 组

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS_FULL} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  每月交易日: {N_DAYS_PER_MONTH} 天")
print(f"  分组数量: {N_GROUPS} 组")

In [ ]:
# ========== 步骤 5: 生成完整数据并计算 IVOL ==========

# 因子收益率序列 (月频)
mkt_premium_monthly = 0.008  # 市场风险溢价 0.8%/月
smb_premium_monthly = 0.003  # 规模溢价
hml_premium_monthly = 0.004  # 价值溢价

# 存储每月结果
monthly_results = []

for month in range(N_MONTHS):
    # --- 生成当月因子收益率 (日频) ---
    mkt_daily = np.random.normal(mkt_premium_monthly/21, 0.012, N_DAYS_PER_MONTH)
    smb_daily = np.random.normal(smb_premium_monthly/21, 0.008, N_DAYS_PER_MONTH)
    hml_daily = np.random.normal(hml_premium_monthly/21, 0.007, N_DAYS_PER_MONTH)
    
    # --- 每只股票的 IVOL 和预期收益 ---
    # IVOL 在股票间有截面差异
    ivol_values = np.random.lognormal(mean=np.log(0.02), sigma=0.5, size=N_STOCKS_FULL)
    ivol_values = np.clip(ivol_values, 0.005, 0.08)
    
    # Beta 截面差异
    betas_mkt = np.random.normal(1.0, 0.3, N_STOCKS_FULL)
    betas_smb = np.random.normal(0.3, 0.2, N_STOCKS_FULL)
    betas_hml = np.random.normal(0.2, 0.15, N_STOCKS_FULL)
    
    # 个股日收益率 = alpha + beta * factors + epsilon
    # 关键：IVOL 异象 —— 高 IVOL 股票的 alpha 更低
    # 月均 alpha = -0.5 * (ivol - mean_ivol)  → 高 IVOL 负 alpha
    alpha_monthly = -15.0 * (ivol_values - ivol_values.mean())  # 负向关系
    alpha_daily = alpha_monthly / N_DAYS_PER_MONTH
    
    # 生成日收益率
    daily_returns = np.zeros((N_DAYS_PER_MONTH, N_STOCKS_FULL))
    daily_residuals = np.zeros((N_DAYS_PER_MONTH, N_STOCKS_FULL))
    
    for i in range(N_STOCKS_FULL):
        eps = np.random.normal(0, ivol_values[i], N_DAYS_PER_MONTH)
        daily_residuals[:, i] = eps
        daily_returns[:, i] = alpha_daily[i] + betas_mkt[i]*mkt_daily + betas_smb[i]*smb_daily + betas_hml[i]*hml_daily + eps
    
    # --- 对每只股票做 FF3 回归，提取 IVOL ---
    ivol_estimated = np.zeros(N_STOCKS_FULL)
    
    for i in range(N_STOCKS_FULL):
        y_i = daily_returns[:, i]
        X_i = np.column_stack([mkt_daily, smb_daily, hml_daily])
        X_i = sm.add_constant(X_i)
        model_i = sm.OLS(y_i, X_i).fit()
        ivol_estimated[i] = np.std(model_i.resid, ddof=3)  # df = T - 4
    
    # --- 计算当月收益率 (月频，用于分组检验) ---
    monthly_returns = daily_returns.sum(axis=0)  # 简单加总
    
    # --- 按 IVOL 分 5 组 ---
    ivol_ranks = pd.qcut(ivol_estimated, N_GROUPS, labels=['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)'])
    
    # --- 各组等权收益率 ---
    for g in range(N_GROUPS):
        group_name = ['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)'][g]
        mask = ivol_ranks == group_name
        group_ret = monthly_returns[mask].mean()
        monthly_results.append({
            'month': month,
            'group': group_name,
            'return': group_ret,
            'n_stocks': mask.sum()
        })

df_results = pd.DataFrame(monthly_results)
print(f"\n📊 步骤 5: 完整模拟完成")
print(f"  总记录数: {len(df_results)}")
print(f"  每月每组平均股票数: {df_results['n_stocks'].mean():.1f}")

---

## 5. IVOL 异象检验：分组收益率分析

### 📐 IVOL 投资组合 (FMP)

$$\text{IVOL Spread} = R_{Q1(\text{Low IVOL})} - R_{Q5(\text{High IVOL})}$$

如果 IVOL 异象存在，Spread 应显著为正（低 IVOL 组合收益更高）。

In [ ]:
# ========== 步骤 6: 各组平均收益率 ==========
group_avg = df_results.groupby('group')['return'].agg(['mean', 'std'])
group_avg['se'] = group_avg['std'] / np.sqrt(N_MONTHS)
group_avg['t_stat'] = group_avg['mean'] / group_avg['se']
group_avg['p_value'] = 2 * (1 - stats.t.cdf(np.abs(group_avg['t_stat']), df=N_MONTHS-1))

# 重新排序
group_order = ['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)']
group_avg = group_avg.reindex(group_order)

print("📊 步骤 6: IVOL 五分组收益率检验")
print(f"{'组别':>12} {'月均收益%':>10} {'标准误%':>8} {'t 值':>8} {'p 值':>10}")
print("-" * 55)
for g in group_order:
    row = group_avg.loc[g]
    print(f"{g:>12} {row['mean']*100:>9.3f}% {row['se']*100:>7.3f}% {row['t_stat']:>8.3f} {row['p_value']:>10.6f}")

# Spread
spread_mean = group_avg.loc['Q1(Low)', 'mean'] - group_avg.loc['Q5(High)', 'mean']
print(f"\n🎯 IVOL Spread (Q1 - Q5): {spread_mean*100:.3f}% 月均")

In [ ]:
# ========== 步骤 7: Spread 时间序列 T 检验 ==========

# 计算每月的 Spread
pivot = df_results.pivot_table(values='return', index='month', columns='group', aggfunc='mean')
pivot = pivot[group_order]
monthly_spreads = pivot['Q1(Low)'] - pivot['Q5(High)']

# T 检验
spread_arr = monthly_spreads.values
spread_mean_ts = spread_arr.mean()
spread_std_ts = spread_arr.std(ddof=1)
spread_se_ts = spread_std_ts / np.sqrt(N_MONTHS)
t_stat_manual = spread_mean_ts / spread_se_ts
p_value_manual = 2 * (1 - stats.t.cdf(abs(t_stat_manual), df=N_MONTHS-1))

print("📊 步骤 7: IVOL Spread 时间序列 T 检验")
print(f"  Spread 月均收益 = {spread_mean_ts*100:.3f}%")
print(f"  Spread 标准差   = {spread_std_ts*100:.3f}%")
print(f"  标准误 (SE)     = {spread_se_ts*100:.3f}%")
print(f"  手算 T 统计量   = {t_stat_manual:.4f}")
print(f"  手算 P 值       = {p_value_manual:.6f}")

# scipy 验证
t_scipy, p_scipy = stats.ttest_1samp(spread_arr, 0)
print(f"\n🔬 scipy 验证:")
print(f"  scipy T 统计量  = {t_scipy:.4f}")
print(f"  scipy P 值      = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

if p_scipy < 0.05:
    print(f"  🎯 结论: IVOL Spread 在 5% 水平下显著 → IVOL 异象存在")
else:
    print(f"  ⚠️ 结论: IVOL Spread 不显著 → 需要更多数据或不同设定")

In [ ]:
# ========== 步骤 8: 单调性检验 (Spearman) ==========

group_ranks = np.arange(1, N_GROUPS + 1)
group_ret_values = np.array([group_avg.loc[g, 'mean'] for g in group_order])

spearman_corr, spearman_p = stats.spearmanr(group_ranks, group_ret_values)

print("📊 步骤 8: 单调性检验 (Spearman Rank Correlation)")
print(f"  组别排序: {list(group_order)}")
print(f"  月均收益: {[f'{r*100:.3f}%' for r in group_ret_values]}")
print(f"  Spearman ρ = {spearman_corr:.4f}")
print(f"  P 值       = {spearman_p:.6f}")

if abs(spearman_corr) > 0.8:
    print(f"  💡 |ρ| > 0.8: 强单调性，IVOL 因子质量优秀")
elif abs(spearman_corr) > 0.5:
    print(f"  💡 |ρ| > 0.5: 中等单调性")
else:
    print(f"  ⚠️ |ρ| < 0.5: 单调性不足，因子效果一般")

In [ ]:
# ========== 步骤 9: Sharpe Ratio ==========

sr_monthly = spread_mean_ts / spread_std_ts
sr_annual = sr_monthly * np.sqrt(12)

print("📊 步骤 9: IVOL Spread 的 Sharpe Ratio")
print(f"  月均 Sharpe Ratio = {sr_monthly:.4f}")
print(f"  年化 Sharpe Ratio = {sr_annual:.4f}")
print(f"\n  📐 验证: t = SR_monthly × √T = {sr_monthly:.4f} × √{N_MONTHS} = {sr_monthly * np.sqrt(N_MONTHS):.4f}")
print(f"  实际 t 值 = {t_stat_manual:.4f}")
print(f"  💡 年化 SR > 0.5 表示因子有经济意义")

In [ ]:
# ========== 可视化: IVOL 五分组收益率 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 左图: 各组月均收益率柱状图
ax = axes[0]
means = [group_avg.loc[g, 'mean']*100 for g in group_order]
colors_bar = ['#2ecc71' if m > 0 else '#e74c3c' for m in means]
bars = ax.bar(range(N_GROUPS), means, color=colors_bar, edgecolor='black', alpha=0.8)
ax.set_xticks(range(N_GROUPS))
ax.set_xticklabels(['Q1\n(Low)', 'Q2', 'Q3', 'Q4', 'Q5\n(High)'], fontsize=10)
ax.set_xlabel('IVOL Group', fontsize=12)
ax.set_ylabel('Monthly Return (%)', fontsize=12)
ax.set_title('Average Monthly Return by IVOL Group', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='black', lw=0.8)
ax.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, means):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.02 if v >= 0 else -0.02
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
            f'{v:.2f}%', ha='center', va=va, fontweight='bold', fontsize=10)

# 中图: Spread 时间序列
ax = axes[1]
ax.plot(range(N_MONTHS), monthly_spreads.values*100, color='steelblue', lw=1.2)
ax.axhline(y=spread_mean_ts*100, color='red', ls='--', lw=1.5, label=f'Mean={spread_mean_ts*100:.2f}%')
ax.axhline(y=0, color='black', lw=0.8)
ax.fill_between(range(N_MONTHS), 0, monthly_spreads.values*100,
                where=monthly_spreads.values > 0, alpha=0.3, color='#2ecc71')
ax.fill_between(range(N_MONTHS), 0, monthly_spreads.values*100,
                where=monthly_spreads.values < 0, alpha=0.3, color='#e74c3c')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('IVOL Spread (%)', fontsize=12)
ax.set_title('Monthly IVOL Spread (Low IVOL - High IVOL)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# 右图: 各组累计收益
ax = axes[2]
cum_returns = (1 + pivot).cumprod()
for g, color in zip(group_order, plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, N_GROUPS))):
    ax.plot(range(N_MONTHS), cum_returns[g].values, label=g, lw=1.5, color=color)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Cumulative Return', fontsize=12)
ax.set_title('Cumulative Return by IVOL Group', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图：IVOL 五分组的月均收益率，低 IVOL 组 (Q1) 收益更高")
print("  中图：每月 IVOL Spread 的时间序列，绿色=正 Spread，红色=负 Spread")
print("  右图：各组的累计收益曲线，低 IVOL 组增长最快")

---

## 6. Fama-MacBeth 回归检验 IVOL 异象

### 📐 Fama-MacBeth 两步法

**第一步**：对每只股票做时序回归，得到 $\beta_i$ 和 $\text{IVOL}_i$

**第二步**：在每个月 $t$ 做截面回归：

$$r_{i,t} = \lambda_{0,t} + \lambda_{1,t} \beta_{i,t-1} + \lambda_{2,t} \text{IVOL}_{i,t-1} + \eta_{i,t}$$

其中 $\lambda_{2,t}$ 的均值和 t 值即为 IVOL 异象的检验统计量。

In [ ]:
# ========== 步骤 10: Fama-MacBeth 回归 ==========

# 使用之前模拟的数据，对每个月做截面回归
fm_lambdas = []

for month in range(N_MONTHS):
    # 重新生成当月数据以获取 beta 和 ivol
    mkt_daily = np.random.normal(mkt_premium_monthly/21, 0.012, N_DAYS_PER_MONTH)
    smb_daily = np.random.normal(smb_premium_monthly/21, 0.008, N_DAYS_PER_MONTH)
    hml_daily = np.random.normal(hml_premium_monthly/21, 0.007, N_DAYS_PER_MONTH)
    
    ivol_values = np.random.lognormal(mean=np.log(0.02), sigma=0.5, size=N_STOCKS_FULL)
    ivol_values = np.clip(ivol_values, 0.005, 0.08)
    
    betas_mkt = np.random.normal(1.0, 0.3, N_STOCKS_FULL)
    
    alpha_monthly = -15.0 * (ivol_values - ivol_values.mean())
    alpha_daily = alpha_monthly / N_DAYS_PER_MONTH
    
    daily_returns = np.zeros((N_DAYS_PER_MONTH, N_STOCKS_FULL))
    for i in range(N_STOCKS_FULL):
        eps = np.random.normal(0, ivol_values[i], N_DAYS_PER_MONTH)
        daily_returns[:, i] = alpha_daily[i] + betas_mkt[i]*mkt_daily + eps
    
    # 估计 IVOL 和 beta
    ivol_est = np.zeros(N_STOCKS_FULL)
    beta_est = np.zeros(N_STOCKS_FULL)
    for i in range(N_STOCKS_FULL):
        model_i = sm.OLS(daily_returns[:, i], sm.add_constant(mkt_daily)).fit()
        ivol_est[i] = np.std(model_i.resid, ddof=2)
        beta_est[i] = model_i.params[1]
    
    # 月收益率
    monthly_ret = daily_returns.sum(axis=0)
    
    # 截面回归: r_i = lambda_0 + lambda_1 * beta_i + lambda_2 * IVOL_i + eta_i
    X_cs = np.column_stack([beta_est, ivol_est])
    X_cs = sm.add_constant(X_cs)
    model_cs = sm.OLS(monthly_ret, X_cs).fit()
    
    fm_lambdas.append({
        'month': month,
        'lambda_0': model_cs.params[0],
        'lambda_beta': model_cs.params[1],
        'lambda_ivol': model_cs.params[2]
    })

df_fm = pd.DataFrame(fm_lambdas)

# T 检验
for var in ['lambda_beta', 'lambda_ivol']:
    arr = df_fm[var].values
    mean_val = arr.mean()
    se_val = arr.std(ddof=1) / np.sqrt(N_MONTHS)
    t_val = mean_val / se_val
    p_val = 2 * (1 - stats.t.cdf(abs(t_val), df=N_MONTHS-1))
    
    label = 'Beta' if 'beta' in var else 'IVOL'
    print(f"📊 Fama-MacBeth λ_{label}:")
    print(f"  月均 λ = {mean_val:.6f}")
    print(f"  t 值   = {t_val:.4f}")
    print(f"  p 值   = {p_val:.6f}")
    if p_val < 0.05:
        print(f"  🎯 显著！")
    print()

---

## 7. 错误定价 × IVOL 双重排序

### 📖 书中的实证方法

> 为了检验特质性波动率异象，在每个错误定价组中，使用最低 IVOL 和最高 IVOL 组之差构建 IVOL 投资组合。

Stambaugh et al. (2015) 提出：IVOL 异象在**被高估的股票中更强**，因为套利不对称性。

### 📐 双重排序步骤

1. 用综合错误定价变量将股票分成 5 组（最被低估 → 最被高估）
2. 在每个错误定价组内，按 IVOL 再分 5 组
3. 在每个错误定价组内，计算 Low IVOL - High IVOL 的 Spread
4. 检验 Spread 是否在被高估组中更显著

In [ ]:
# ========== 步骤 11: 双重排序模拟 ==========

N_MISPRICING_GROUPS = 5
double_sort_results = []

for month in range(N_MONTHS):
    # 生成数据
    mkt_daily = np.random.normal(mkt_premium_monthly/21, 0.012, N_DAYS_PER_MONTH)
    
    # 错误定价变量 (综合得分, 越高越被高估)
    mispricing_score = np.random.normal(0, 1, N_STOCKS_FULL)
    
    # IVOL
    ivol_values = np.random.lognormal(mean=np.log(0.02), sigma=0.5, size=N_STOCKS_FULL)
    ivol_values = np.clip(ivol_values, 0.005, 0.08)
    
    # 收益率: IVOL 异象在被高估股票中更强
    # alpha = -15 * ivol * (1 + 0.5 * mispricing_score_normalized)
    mispricing_norm = (mispricing_score - mispricing_score.mean()) / mispricing_score.std()
    alpha_monthly = -15.0 * ivol_values * (1 + 0.5 * mispricing_norm)
    alpha_daily = alpha_monthly / N_DAYS_PER_MONTH
    
    daily_returns = np.zeros((N_DAYS_PER_MONTH, N_STOCKS_FULL))
    for i in range(N_STOCKS_FULL):
        eps = np.random.normal(0, ivol_values[i], N_DAYS_PER_MONTH)
        daily_returns[:, i] = alpha_daily[i] + 1.0 * mkt_daily + eps
    
    # 估计 IVOL
    ivol_est = np.zeros(N_STOCKS_FULL)
    for i in range(N_STOCKS_FULL):
        model_i = sm.OLS(daily_returns[:, i], sm.add_constant(mkt_daily)).fit()
        ivol_est[i] = np.std(model_i.resid, ddof=2)
    
    monthly_ret = daily_returns.sum(axis=0)
    
    # 错误定价分组
    mispricing_groups = pd.qcut(mispricing_score, N_MISPRICING_GROUPS,
                                labels=['Most Underpriced', '2', '3', '4', 'Most Overpriced'])
    
    # 在每个错误定价组内按 IVOL 分组
    for mp_group in ['Most Underpriced', '2', '3', '4', 'Most Overpriced']:
        mask_mp = mispricing_groups == mp_group
        ivol_sub = ivol_est[mask_mp]
        ret_sub = monthly_ret[mask_mp]
        
        # IVOL 分 5 组
        ivol_sub_groups = pd.qcut(ivol_sub, N_GROUPS, labels=['Low', '2', '3', '4', 'High'])
        
        low_ret = ret_sub[ivol_sub_groups == 'Low'].mean()
        high_ret = ret_sub[ivol_sub_groups == 'High'].mean()
        spread = low_ret - high_ret
        
        double_sort_results.append({
            'month': month,
            'mispricing_group': mp_group,
            'spread': spread
        })

df_double = pd.DataFrame(double_sort_results)

# 汇总
double_summary = df_double.groupby('mispricing_group')['spread'].agg(['mean', 'std'])
double_summary['se'] = double_summary['std'] / np.sqrt(N_MONTHS)
double_summary['t_stat'] = double_summary['mean'] / double_summary['se']
double_summary['p_value'] = [2*(1-stats.t.cdf(abs(t), df=N_MONTHS-1)) for t in double_summary['t_stat']]

mp_order = ['Most Underpriced', '2', '3', '4', 'Most Overpriced']
double_summary = double_summary.reindex(mp_order)

print("📊 步骤 11: 错误定价 × IVOL 双重排序检验")
print(f"{'错误定价组':>18} {'Spread%':>10} {'t 值':>8} {'p 值':>10}")
print("-" * 50)
for g in mp_order:
    row = double_summary.loc[g]
    print(f"{g:>18} {row['mean']*100:>9.3f}% {row['t_stat']:>8.3f} {row['p_value']:>10.6f}")

In [ ]:
# ========== 可视化: 双重排序热力图 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 各错误定价组的 IVOL Spread
ax = axes[0]
spreads = [double_summary.loc[g, 'mean']*100 for g in mp_order]
t_vals = [double_summary.loc[g, 't_stat'] for g in mp_order]
colors_bar = ['#2ecc71' if s > 0 else '#e74c3c' for s in spreads]
bars = ax.bar(range(N_MISPRICING_GROUPS), spreads, color=colors_bar, edgecolor='black', alpha=0.8)
ax.set_xticks(range(N_MISPRICING_GROUPS))
ax.set_xticklabels(['Underpriced\n(Most)', '2', '3', '4', 'Overpriced\n(Most)'], fontsize=9)
ax.set_xlabel('Mispricing Group', fontsize=12)
ax.set_ylabel('IVOL Spread (%)', fontsize=12)
ax.set_title('IVOL Spread by Mispricing Group', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='black', lw=0.8)
ax.grid(axis='y', alpha=0.3)
for bar, v, t in zip(bars, spreads, t_vals):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.03 if v >= 0 else -0.03
    sig = '***' if abs(t) > 2.58 else ('**' if abs(t) > 1.96 else ('*' if abs(t) > 1.645 else ''))
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
            f'{v:.2f}%\n{sig}', ha='center', va=va, fontweight='bold', fontsize=9)

# 右图: 行为金融学解释示意图
ax = axes[1]
ax.axis('off')
ax.text(0.5, 0.95, 'Stambaugh et al. (2015): Arbitrage Asymmetry',
        ha='center', va='top', fontsize=14, fontweight='bold', transform=ax.transAxes)

explanation = [
    (0.5, 0.80, 'Mispricing Direction', 13, 'bold'),
    (0.25, 0.70, 'Underpriced', 12, 'normal'),
    (0.75, 0.70, 'Overpriced', 12, 'normal'),
    (0.25, 0.60, 'Arbitrage: Buy', 11, 'normal'),
    (0.75, 0.60, 'Arbitrage: Short', 11, 'normal'),
    (0.25, 0.50, 'Less limited', 11, 'normal'),
    (0.75, 0.50, 'More limited', 11, 'normal'),
    (0.5, 0.35, '→ IVOL effect WEAKER in underpriced', 11, 'normal'),
    (0.5, 0.25, '→ IVOL effect STRONGER in overpriced', 12, 'bold'),
    (0.5, 0.10, 'Result: Negative IVOL-return relation', 13, 'bold'),
]
for x, y, text, fs, fw in explanation:
    ax.text(x, y, text, ha='center', va='center', fontsize=fs, fontweight=fw,
            transform=ax.transAxes,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8) if fw == 'bold' else None)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图：不同错误定价组中的 IVOL Spread，被高估组的 Spread 更大")
print("  右图：Stambaugh et al. (2015) 的套利不对称性理论")
print("  核心逻辑：卖空限制 → 被高估股票套利不充分 → IVOL 异象更强")

---

## 8. A 股 IVOL 异象的特殊性

### 📖 书中的 A 股实证结论

> 在最被高估档内的 5 个 IVOL 组中，IVOL 和收益率呈现出负相关。然而，在最被低估的档内的 5 个 IVOL 组中，并没有观察到和美股一样的正相关。

A 股市场的 IVOL 异象有几个特点：

| 特征 | 美股 (Ang et al.) | A 股 |
|------|-------------------|------|
| IVOL-收益关系 | 负相关 | 负相关 |
| 被低估股票 | IVOL 正相关 | 无显著正相关 |
| 被高估股票 | IVOL 负相关更强 | IVOL 负相关更强 |
| 市值加权影响 | 显著性降低 | 显著性降低 |
| Liu-Shi-Lian 模型 | - | 市值加权下不显著 |

### 💡 A 股特殊性的可能解释

- **高噪音**：A 股市场散户占比高，噪声交易者行为更极端
- **高换手率**：A 股换手率远高于成熟市场
- **卖空限制**：A 股卖空机制更受限，套利不对称性更强

---

## 📌 核心概念回顾

### 📌 特质波动率 (IVOL, Idiosyncratic Volatility)
- **定义**: 多因子模型回归残差的标准差
- **公式**: $\text{IVOL}_i = \text{std}(\hat{\varepsilon}_{i,t})$
- **含义**: 衡量个股无法被因子解释的波动程度
- **计算**: FF3 时序回归 → 残差 → 标准差

### 📌 IVOL 异象 (IVOL Anomaly)
- **定义**: 低 IVOL 股票的未来收益率高于高 IVOL 股票
- **公式**: $\text{IVOL Spread} = R_{Q1(\text{Low})} - R_{Q5(\text{High})} > 0$
- **含义**: 违反经典理论（IVOL 应与收益无关）
- **判断标准**: Spread 的 t 值 > 2.0 表示显著

### 📌 套利不对称性 (Arbitrage Asymmetry)
- **定义**: 投资者愿意买入被低估股票，但不愿卖空被高估股票
- **含义**: 被高估股票的错误定价难以消除
- **结果**: IVOL 异象在被高估股票中更强

### 📌 Lottery Demand
- **定义**: 投资者偏好高 IVOL 股票（像买彩票一样）
- **含义**: 高 IVOL 股票被过度追捧，价格被推高
- **结果**: 高 IVOL 股票的未来收益率反而更低

### 🔗 完整流程
```
日频数据 → FF3 时序回归 → 提取残差 → 计算 IVOL
                                    ↓
              按 IVOL 分组 → 计算各组月均收益 → T 检验
                                    ↓
              Fama-MacBeth 回归 → λ_IVOL 显著性
                                    ↓
              双重排序 (错误定价 × IVOL) → 套利不对称性
```

### 📝 检验指标汇总

| 指标 | 公式 | 含义 | 阈值 |
|------|------|------|------|
| IVOL Spread | $R_{Q1} - R_{Q5}$ | 多空收益差 | > 0 表示异象存在 |
| T 统计量 | $\bar{s} / SE(\bar{s})$ | Spread 显著性 | > 1.96 (5%) |
| Sharpe Ratio | $\bar{s} / \sigma_s$ | 风险调整收益 | > 0.5 (年化) |
| Spearman ρ | rank 相关 | 单调性 | > 0.8 强单调 |
| FM λ_IVOL | 截面回归系数 | IVOL 的边际效应 | < 0 表示负相关 |

---

## ❌ 常见误区

### ❌ 误区 1: IVOL 就是总波动率
**✓ 正确理解**: IVOL 是回归**残差**的波动率，不是股票收益率的总波动率。总波动率 = 系统性波动 + 特质性波动。IVOL 只包含后者。

### ❌ 误区 2: IVOL 异象意味着高 IVOL 股票不值得投资
**✓ 正确理解**: IVOL 异象是一个**统计规律**，不代表因果关系。高 IVOL 股票可能因为 lottery demand 被高估，但也可能在某些时期表现优异。异象的存在与市场情绪、套利限制等因素相关。

### ❌ 误区 3: 用未来数据计算 IVOL（前视偏差）
**✓ 正确理解**: IVOL 必须使用**过去**的收益率数据计算。例如，用第 t 月的 IVOL 预测第 t+1 月的收益率。绝不能用第 t+1 月的数据计算第 t 月的 IVOL。

### ❌ 误区 4: IVOL 与 Beta 是同一回事
**✓ 正确理解**: Beta 衡量的是**系统性风险暴露**（因子载荷），而 IVOL 衡量的是**特质性风险**（残差波动）。两者概念完全不同。低 Beta 异象和低 IVOL 异象是两个独立的异象。

### ❌ 误区 5: IVOL 异象在所有市场都一样
**✓ 正确理解**: IVOL 异象的强度和表现因市场而异。A 股的 IVOL 异象与美股有显著差异——在被低估股票中未观察到正相关，且受市值加权和定价模型选择的影响更大。